# Maximum A Posteriori (MAP) Estimation

Three Bayesians find the same coin and see the same 8 heads in 10 tosses — yet end with three different answers: **0.501**, **0.607**, and **0.8**. The difference is their **priors**: adamantly-fair, mildly-fair, and non-informative. Updating a prior with data yields the **posterior** (belief *after* seeing data), and reporting its peak — its mode — is **maximum a posteriori estimation**. And the third answer reveals the punchline: MAP with a flat prior *is* the frequentist/MLE answer. It is based on the ninth video of Lesson 6 (Point Estimation).

## 1. Introduction

The last notebook introduced priors philosophically; this one watches them work. Three statisticians, one dataset, three belief curves evolving — then a rule for compressing an entire belief curve into one number.

After completing this notebook you will be able to:

- Model priors of different strengths (narrow, wide, flat/non-informative) and update them with data.
- Track belief curves after 1 toss and after 10, reproducing the video's three trajectories.
- Define **posterior** and **MAP** (the posterior's mode) and compute the three answers 0.501 / 0.607 / 0.8.
- State and verify the key identity: **MAP with a uniform prior = MLE**.

## 2. Theory

### Three Bayesians, three priors

All three want the coin's heads-probability $p$:

1. **The conservative:** absolutely convinced the coin is fair — a **very narrow** prior centered at 0.5, dropping to zero quickly in both directions.
2. **The moderate:** thinks the coin should be fair but is willing to believe in bias — also peaked at 0.5, but **more spread out**.
3. **The agnostic:** doesn't want to assume anything — the same weight on every value of $p$: a **non-informative (uniform) prior**, adding no information.

### One toss: heads

- The conservative's curve doesn't visibly move.
- The moderate updates *slightly* — it is only one observation.
- The agnostic changes **drastically**: from flat to a belief that grows linearly in $p$ (small $p$ now implausible — it just produced a head!).

### Ten tosses: 8 heads, 2 tails

- The conservative is still glued to 0.5 — the curve has barely moved.
- The moderate's curve is **narrower and shifted**, now peaking around 0.6–0.65.
- The agnostic went from knowing nothing to being fairly sure $p \approx 0.8$.

Same data, different priors ⇒ different final beliefs.

### Posterior, and the MAP estimate

Terminology: the original belief is the **prior** (before data); the updated belief is the **posterior** (after data). The belief curve is rich — it scores *every* possible value — but often one representative number is wanted. Among several criteria, the most used: take the value with the **highest posterior probability** — the **mode** of the posterior. This is **maximum a posteriori estimation (MAP)**.

The three MAP answers for 8H2T:

$$
\text{conservative: } 0.501 \qquad \text{moderate: } 0.607 \qquad \text{agnostic: } 0.8
$$

### The punchline: flat prior ⇒ MLE

The agnostic's 0.8 *is* the frequentist answer — and always will be: **MAP with a uniform prior is identical to maximum likelihood** (maximizing prior × likelihood with a constant prior is maximizing the likelihood). Priors are what make Bayesian estimation different; show up without meaningful beliefs and MAP is a cumbersome way to do frequentist statistics.

## 3. Worked Examples

### Example 1: priors as phantom flips

Model the fair-centered priors as $\propto p^{a}(1-p)^{a}$ ("$2a$ phantom fair flips"). Then the posterior after $k$ heads in $n$ tosses is $\propto p^{a+k}(1-p)^{a+n-k}$, with mode

$$
\hat{p}_{\text{MAP}} = \frac{a + k}{2a + n}
$$

Matching the video's answers for 8H2T: the conservative's $0.501$ needs $a \approx 1500$ (thousands of phantom flips!), the moderate's $0.607$ needs $a \approx 9$, and the agnostic has $a = 0$: $\hat p = \frac{8}{10} = 0.8$ — the MLE.

### Example 2: after a single head

Posteriors $\propto p^{a+1}(1-p)^{a}$: the agnostic's ($a=0$) is exactly the linear function $p$ — flat belief turned into "the larger $p$, the likelier," precisely the video's description. Its mode is already at $p = 1$ (one head, no tails — the MLE edge case), while both fair-centered priors keep their modes near 0.5.

### Example 3: reading a belief curve

The posterior shows *how strongly every value is believed*: the moderate's curve at 0.607 is noticeably wide — the honest summary is "around 0.6, give or take 0.1," a nuance the single MAP number discards. (Other one-number summaries — posterior mean, median — are the reading mentioned at the section's end.)

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

p = np.linspace(0.001, 0.999, 1999)

bayesians = {
    "conservative (a = 1500)": 1500,
    "moderate (a = 9)": 9,
    "agnostic (a = 0, uniform)": 0,
}

def posterior(a, heads, tails):
    curve = p**(a + heads) * (1 - p)**(a + tails)
    return curve / np.trapezoid(curve, p)

def map_estimate(a, heads, tails):
    return (a + heads) / (2 * a + heads + tails)

### The three MAP answers for 8H2T

In [ ]:
for name, a in bayesians.items():
    est = map_estimate(a, 8, 2)
    print(f"{name:<26} MAP = {est:.3f}")
print("\nvideo: 0.501, 0.607, 0.8 -- and the uniform-prior answer equals the MLE.")

### MAP with a flat prior = MLE, checked on many datasets

In [ ]:
rng = np.random.default_rng(0)

ok = True
for _ in range(10):
    n = int(rng.integers(3, 60))
    k = int(rng.integers(0, n + 1))
    post = p**k * (1 - p)**(n - k)          # flat prior: posterior = likelihood
    ok &= abs(p[np.argmax(post)] - (k / n if 0 < k < n else (0.0 if k == 0 else 1.0))) < 1e-3
print(f"argmax of (flat-prior posterior) == k/n in all datasets: {ok}")
print("With a constant prior, maximizing prior x likelihood IS maximizing the")
print("likelihood -- MAP degenerates gracefully into MLE.")

## 5. Visualizations

### The video's story: three belief curves, three moments

Rows: prior → after 1 head → after 8H2T. Watch who moves and by how much.

In [ ]:
stages = [("prior", 0, 0), ("after 1 toss (H)", 1, 0), ("after 8H 2T", 8, 2)]

fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharex=True)
colors = ["tab:red", "tab:orange", "tab:green"]
for col, ((name, a), color) in enumerate(zip(bayesians.items(), colors)):
    for row, (stage, h, t) in enumerate(stages):
        ax = axes[row, col]
        curve = posterior(a, h, t)
        ax.plot(p, curve, color=color, linewidth=2)
        mode = map_estimate(a, h, t) if (a + h) > 0 or (a + t) > 0 else 0.5
        if row == 2:
            ax.axvline(mode, color="black", linestyle=":", linewidth=1.5)
            ax.set_title(f"MAP = {mode:.3f}", fontsize=11)
        if row == 0:
            ax.set_title(name, fontsize=11)
        if col == 0:
            ax.set_ylabel(stage)
        ax.set_yticks([])
        ax.grid(True, alpha=0.3)
for ax in axes[2]:
    ax.set_xlabel("p")
plt.suptitle("Same data, different priors, different final beliefs (video's three Bayesians)")
plt.tight_layout()
plt.show()

### The agnostic's first update: flat to linear

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
axes[0].plot(p, posterior(0, 0, 0), color="tab:green", linewidth=2)
axes[0].set_title("agnostic prior: every p equally believable")
axes[1].plot(p, posterior(0, 1, 0), color="tab:green", linewidth=2)
axes[1].set_title("after one head: belief grows linearly with p")
for ax in axes:
    ax.set_xlabel("p")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("belief density")
plt.tight_layout()
plt.show()

### The whole spectrum: MAP vs prior strength

The three Bayesians are three points on one curve — MAP $= \frac{a+8}{2a+10}$ sliding from the MLE (0.8) down to the immovable 0.5.

In [ ]:
a_grid = np.logspace(-2, 4, 300)
maps = (a_grid + 8) / (2 * a_grid + 10)

plt.figure(figsize=(9, 4.5))
plt.plot(a_grid, maps, color="tab:purple", linewidth=2)
for (name, a), color in zip(bayesians.items(), colors):
    a_plot = max(a, 1e-2)
    plt.plot(a_plot, map_estimate(a, 8, 2), "o", markersize=10, color=color,
             label=f"{name}: {map_estimate(a, 8, 2):.3f}")
plt.axhline(0.8, color="gray", linestyle="--", linewidth=1, label="MLE 0.8")
plt.xscale("log")
plt.xlabel("prior strength a (log)")
plt.ylabel("MAP estimate after 8H 2T")
plt.title("MAP interpolates between the MLE and the prior's center")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

### The whole story on one page

Everything above in a single sheet: three Bayesians' beliefs before and after the flips, their MAP estimates, priors as phantom flips, and why a flat prior gives the MLE.

<img src="img/08_map.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** Compute the three MAP estimates if the data had been 5 heads, 5 tails. Why do all three now agree (nearly)?

**Exercise 2.** Define prior, posterior, and MAP in one sentence each, and name the criterion MAP uses to compress a belief curve into a number.

### Intermediate

**Exercise 3.** Reproduce the video's numbers: find (numerically) the prior strengths $a$ whose 8H2T MAP equals 0.501 and 0.607 exactly. How many "phantom flips" does each represent?

**Exercise 4.** The posterior is not just its peak: for the moderate ($a = 9$), compute the posterior's mode, mean, and the central interval containing 90% of the belief. How different are mode and mean, and why (which way does the curve lean)?

### Advanced

**Exercise 5.** More data melts the prior: for all three Bayesians, compute the MAP after $n = 10, 100, 1000, 10{,}000$ tosses of a true-0.8 coin (use expected counts $k = 0.8n$). Which prior takes longest to surrender, and does even $a = 1500$ eventually reach 0.8?

**Exercise 6.** A *wrongly-centered* prior: a fourth Bayesian is convinced the coin favors tails ($\propto p^{2}(1-p)^{20}$, peaked near 0.09). Track their MAP after 8H2T and after 80H20T. What does this show about the difference between a strong prior and a wrong prior?

## 7. Solutions

In [ ]:
# Exercise 1: balanced data
for name, a in bayesians.items():
    print(f"{name:<26} MAP = {map_estimate(a, 5, 5):.3f}")
print("All exactly 0.5: the evidence agrees with the fair-centered priors, and")
print("the uniform prior's MLE is 5/10 = 0.5 too -- no tension, no disagreement.")

In [ ]:
# Exercise 2: definitions
print("Exercise 2:")
print("Prior: your belief distribution over the parameter BEFORE seeing data.")
print("Posterior: the updated belief distribution AFTER seeing the data.")
print("MAP: the point estimate taking the posterior's MODE -- the single value")
print("with the highest posterior belief.")

In [ ]:
# Exercise 3: reverse-engineering the video's priors
# MAP = (a + 8)/(2a + 10) = m  =>  a = (8 - 10 m)/(2 m - 1)
for target in [0.501, 0.607]:
    a_t = (8 - 10 * target) / (2 * target - 1)
    print(f"MAP = {target}: a = {a_t:,.0f}  ({2 * a_t:,.0f} phantom fair flips)")
print("The conservative behaves as if they'd already watched ~3,000 fair flips;")
print("the moderate, ~18.")

In [ ]:
# Exercise 4: mode vs mean vs interval for the moderate
post = posterior(9, 8, 2)
mode = p[np.argmax(post)]
mean = np.trapezoid(p * post, p)
cdf = np.cumsum(post) * (p[1] - p[0])
lo, hi = p[np.searchsorted(cdf, 0.05)], p[np.searchsorted(cdf, 0.95)]
print(f"Exercise 4: mode = {mode:.3f}, mean = {mean:.3f}, 90% interval = "
      f"[{lo:.3f}, {hi:.3f}]")
print("The mean sits slightly below the mode: the posterior leans left (the")
print("(1-p)^11 factor gives it a longer tail toward small p), and means chase")
print("tails while modes ignore them.")

In [ ]:
# Exercise 5: the priors surrender at different speeds
print("Exercise 5: MAP with expected counts from a true-0.8 coin")
print(f"{'n':>7}" + "".join(f"{name.split()[0]:>15}" for name in bayesians))
for n in [10, 100, 1000, 10_000]:
    row = f"{n:>7}"
    for name, a in bayesians.items():
        row += f"{map_estimate(a, 0.8 * n, 0.2 * n):>15.3f}"
    print(row)
print("The a = 1500 prior needs thousands of tosses but does converge: MAP ->")
print("k/n as n -> infinity for ANY fixed prior strength.")

In [ ]:
# Exercise 6: a wrong prior vs data
def map_general(ah, at, h, t):
    return (ah + h) / (ah + at + h + t)

print("Exercise 6: tails-favoring prior  ~ p^2 (1-p)^20 (peak ~0.09):")
print(f"  after  8H  2T: MAP = {map_general(2, 20, 8, 2):.3f}")
print(f"  after 80H 20T: MAP = {map_general(2, 20, 80, 20):.3f}")
print("A STRONG prior slows learning; a WRONG prior starts you far from the")
print("truth -- but the same update machinery corrects both, at the price of")
print("more data. Bad beliefs cost evidence.")

## 8. Conclusion

- Three priors — adamant, moderate, non-informative — met the same 8H2T and produced three posteriors: barely moved / shifted and narrowed (peak ≈ 0.6) / confidently near 0.8.
- **Prior** = belief before data; **posterior** = belief after; **MAP** = the posterior's **mode**, the most-used one-number summary of a belief curve. The video's answers: **0.501, 0.607, 0.8**.
- With a fair-centered prior of strength $a$: $\hat p_{\text{MAP}} = \frac{a+k}{2a+n}$ — an interpolation between the prior's center and the MLE, with $2a$ acting as phantom flips.
- **Flat prior ⇒ MAP = MLE**: without meaningful prior beliefs, MAP is a roundabout frequentist computation — priors are exactly what make Bayesian estimation distinct.
- More data melts any fixed prior (even $a = 1500$), and wrong priors are corrected too, at the cost of evidence.

Next: the mathematics of the update itself — Bayes' theorem doing the work behind these curves.